# 02 — Limit-lowering uptake and selected amount

Reproduces Table S4: the proportion of participants who lowered their gambling
limit, and, among those who did, the amount selected.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal, chi2_contingency
from scipy.stats.contingency import association

ARM_COL = "CONTROL_GROUP"
ARM_MAP = {"AA": "A", "BB": "B", "CC": "C", "EE": "E", "FF": "F", "CONTROLGROUP": "Control"}

In [ ]:
def limit_lowering(path, label, interv_arms):
    df = pd.read_csv(path, low_memory=False)
    df["Arm"] = df[ARM_COL].map(ARM_MAP).fillna(df[ARM_COL])

    print("#" * 70); print(f"#  {label}"); print("#" * 70)

    all_arms = interv_arms + ["Control"]
    df["lowered"] = (df["ACTION"] == "SET_LOWER_LIMITS").astype(int)

    print("\n--- Limit-lowering uptake ---")
    tab = pd.crosstab(df["Arm"], df["lowered"]).reindex(all_arms)
    up = df.groupby("Arm")["lowered"].agg(sum="sum", count="count").reindex(all_arms)
    up["pct"] = (100 * up["sum"] / up["count"]).round(1)
    print(up)

    chi2, p, dof, _ = chi2_contingency(tab)
    v = association(tab, method="cramer")
    print(f"\nChi-square: chi2={chi2:.2f}, df={dof}, p={p:.4f}, Cramer's V={v:.3f}")

    print("\n--- Selected limit amount among setters ---")
    setters = df[(df["ACTION"] == "SET_LOWER_LIMITS") & (df["VALUE_ACTION"].notna())]
    amt = setters.groupby("Arm")["VALUE_ACTION"].agg(n="count", median="median").reindex(interv_arms)
    print(amt)

    groups = [setters.loc[setters.Arm == a, "VALUE_ACTION"].dropna().values for a in interv_arms]
    H, p_amt = kruskal(*groups)
    print(f"\nKruskal-Wallis (amount): H={H:.2f}, p={p_amt:.4f}")

limit_lowering("P10_final.csv", "Experiment 1", ["A", "C", "E", "F"])
print()
limit_lowering("P11_final.csv", "Experiment 2", ["A", "B", "C"])